# Notebook 01 — Stratified FPG Regression Models
### XAI for FPG Prediction: Algorithmic Fairness in Health Insurance Underwriting
**Paper §3.1–3.2, §4.1–4.3**

This notebook covers:
1. KNHANES data loading & preprocessing  
2. Demographic stratification (6 age × sex groups)  
3. Stratified model training: LR · Ridge · RF · LGBM · XGB · MLP  
4. Hold-out evaluation + 5-fold cross-validation  
5. Temporal validation (G2-b robustness)  
6. SHAP variable importance  
7. Serialisation of artefacts → used by notebooks 02–05

> **Outputs** saved to `outputs/` directory.


In [1]:
# ── Standard library ────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")
import logging
import os
from pathlib import Path

# ── Numeric / data ───────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import pyreadstat
import joblib
import scipy.stats as stats

# ── ML ───────────────────────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold, train_test_split
import lightgbm as lgb
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── XAI ─────────────────────────────────────────────────────────────────────
import shap

# ── Visualisation ────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Config ───────────────────────────────────────────────────────────────────
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

# Publication figure style
plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "font.size":         11,
    "axes.unicode_minus": False,
    "figure.dpi":        150,
})

SEED  = 42
DPI   = 300
np.random.seed(SEED)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

log.info("All packages loaded successfully.")


2026-04-27 13:43:10,864 | INFO | All packages loaded successfully.


## 1. Constants & Configuration

In [2]:
# ── FPG stage boundaries (ADA 2023) ─────────────────────────────────────────
GLUCOSE_STAGES = {
    "normal":   (0,   100),   # Normal: FPG < 100 mg/dL
    "ifg":      (100, 126),   # Impaired Fasting Glucose: 100–125 mg/dL
    "diabetes": (126, 9999),  # Diabetes: FPG ≥ 126 mg/dL
}

def assign_glucose_stage(fpg: float) -> str:
    """Classify FPG (mg/dL) into clinical glucose stage per ADA (2023)."""
    if fpg < 100:  return "normal"
    if fpg < 126:  return "ifg"
    return "diabetes"


# ── Demographic group configuration ──────────────────────────────────────────
# age_group coding: 0 = Young (19-39), 1 = Middle (40-64), 2 = Elderly (65+)
GROUP_CONFIG = {
    "Young_Male":    {"age_min": 19, "age_max": 39, "sex_code": 1.0, "age_group": 0.0},
    "Young_Female":  {"age_min": 19, "age_max": 39, "sex_code": 2.0, "age_group": 0.0},
    "Middle_Male":   {"age_min": 40, "age_max": 64, "sex_code": 1.0, "age_group": 1.0},
    "Middle_Female": {"age_min": 40, "age_max": 64, "sex_code": 2.0, "age_group": 1.0},
    "Elderly_Male":  {"age_min": 65, "age_max": 99, "sex_code": 1.0, "age_group": 2.0},
    "Elderly_Female":{"age_min": 65, "age_max": 99, "sex_code": 2.0, "age_group": 2.0},
}

GROUP_LABELS = {
    "Young_Male":    "Young Male",
    "Young_Female":  "Young Female",
    "Middle_Male":   "Middle-aged Male",
    "Middle_Female": "Middle-aged Female",
    "Elderly_Male":  "Elderly Male",
    "Elderly_Female":"Elderly Female",
}

# Publication colour palette (colorblind-friendly)
GROUP_COLORS = {
    "Young_Male":    "#1565C0",
    "Young_Female":  "#90CAF9",
    "Middle_Male":   "#E65100",
    "Middle_Female": "#FFCC80",
    "Elderly_Male":  "#2E7D32",
    "Elderly_Female":"#A5D6A7",
}

ALGORITHMS = ["LR", "Ridge", "RF", "LGBM", "XGB", "MLP"]
BASELINE_ALGOS = {"LR", "Ridge"}   # actuarial baselines — excluded from best-model selection

log.info("Constants defined: %d groups, %d algorithms", len(GROUP_CONFIG), len(ALGORITHMS))


2026-04-27 13:43:10,880 | INFO | Constants defined: 6 groups, 6 algorithms


## 2. Data Loading — KNHANES (3 survey years)

In [3]:
# ── Variable lists ──────────────────────────────────────────────────────────
KEY_COLS    = ["ID", "year", "sex", "age"]
TARGET_COLS = ["HE_glu"]   # FPG + diabetes medication flag

CAT_COLS = [
    "HE_obe",  "BO1_1", "BO1_2",  "BO1_3",
    "BD1_11",  "BD2_1", "BS3_1",
    "BE3_71",  "BE3_75", "BE3_81", "BE3_91", "pa_aerobic",
    "L_BR_FQ", "BP1",   "mh_stress",
    "incm",    "ho_incm", "edu",   "BH1",
]
NUM_COLS = [
    "HE_BMI", "HE_wc",  "HE_wt",
    "N_EN",   "N_CHO",  "N_SUGAR", "N_NA",
    "N_FAT",  "N_SFA",  "N_TDF",   "N_K", "N_PROT",
]
ALL_VARS = KEY_COLS + CAT_COLS + NUM_COLS + TARGET_COLS

# ── Load SAS files ───────────────────────────────────────────────────────────
# NOTE: update filenames to match your local KNHANES files
KNHANES_FILES = [
    "hn22_all.sas7bdat",
    "hn23_all.sas7bdat",
    "hn24_all.sas7bdat",
]

frames = []
for fpath in KNHANES_FILES:
    df_yr, _ = pyreadstat.read_sas7bdat(fpath)
    frames.append(df_yr[[v for v in ALL_VARS if v in df_yr.columns]].copy())
    log.info("Loaded %s: %d rows", fpath, len(df_yr))

df_raw = pd.concat(frames, axis=0, ignore_index=True)
log.info("Merged dataset: %d rows × %d columns", *df_raw.shape)


2026-04-27 13:43:11,523 | INFO | Loaded hn22_all.sas7bdat: 6265 rows
2026-04-27 13:43:12,169 | INFO | Loaded hn23_all.sas7bdat: 6929 rows
2026-04-27 13:43:13,079 | INFO | Loaded hn24_all.sas7bdat: 6997 rows
2026-04-27 13:43:13,089 | INFO | Merged dataset: 20191 rows × 36 columns


## 3. Preprocessing

In [4]:
# ── Step 1: Drop missing FPG ────────────────────────────────────────────────
n0 = len(df_raw)
df = df_raw.dropna(subset=["HE_glu"]).reset_index(drop=True)
log.info("Step 1 — Drop missing FPG: %d → %d (removed %d)", n0, len(df), n0 - len(df))

# ── Step 2: Exclude diabetes-medication users ────────────────────────────────
# HE_DMdr = 1: took diabetes medication on the examination day
# Rationale: pharmacological suppression of FPG distorts the outcome variable
# (explicitly stated in paper Methods §3.1)
# ── Step 2: Exclude diabetes-medication users (if column available) ────────────
n1 = len(df)
if "HE_DMdr" in df.columns:
    df = df[df["HE_DMdr"] != 1].reset_index(drop=True)
    log.info("Step 2 — Exclude medicated diabetics: %d → %d (removed %d)",
             n1, len(df), n1 - len(df))
else:
    log.warning("Step 2 — 'HE_DMdr' not found in this dataset "
                "(column absent in selected survey years). Skipping exclusion.")

# ── Step 3: Age ≥ 19 ─────────────────────────────────────────────────────────
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df = df[df["age"] >= 19].reset_index(drop=True)

# ── Step 4: FPG outlier removal (40–400 mg/dL) ──────────────────────────────
n3 = len(df)
df = df[(df["HE_glu"] >= 40) & (df["HE_glu"] <= 400)].reset_index(drop=True)
log.info("Step 4 — FPG outlier removal: %d → %d (removed %d)",
         n3, len(df), n3 - len(df))

# ── Derived columns ──────────────────────────────────────────────────────────
def assign_age_group(age: float) -> float:
    """Encode age band: 0=Young(19-39), 1=Middle(40-64), 2=Elderly(65+)."""
    if age <= 39: return 0.0
    if age <= 64: return 1.0
    return 2.0

df["age_group"]    = df["age"].apply(assign_age_group)
df["GlucoseStage"] = df["HE_glu"].apply(assign_glucose_stage)

# ── Rename columns to English ────────────────────────────────────────────────
COLUMN_MAP = {
    "ID": "ID",           "year": "SurveyYear", "sex": "Sex",
    "age": "Age",         "age_group": "AgeGroup",
    "HE_glu": "FPG",
    "HE_obe": "ObesityStatus",
    "BO1_1": "WeightChangeStatus",  "BO1_2": "WeightLossAmount",
    "BO1_3": "WeightGainAmount",    "BD1_11": "DrinkingFrequency",
    "BD2_1": "DrinkingAmount",      "BS3_1": "SmokingStatus",
    "BE3_71": "VigorousAct_Work",   "BE3_75": "VigorousAct_Leisure",
    "BE3_81": "ModerateAct_Work",   "BE3_91": "WalkingActivity",
    "pa_aerobic": "AerobicRate",    "L_BR_FQ": "BreakfastFreq",
    "BP1": "StressLevel",           "mh_stress": "StressAwareness",
    "incm": "IncomeQuartile",       "ho_incm": "HouseholdIncome",
    "edu": "EducationLevel",        "BH1": "HealthScreening",
    "HE_BMI": "BMI",     "HE_wc": "WaistCirc", "HE_wt": "Weight",
    "N_EN": "Energy_kcal","N_CHO": "Carb_g",    "N_SUGAR": "Sugar_g",
    "N_NA": "Sodium_mg",  "N_FAT": "Fat_g",     "N_SFA": "SatFat_g",
    "N_TDF": "Fiber_g",   "N_K": "Potassium_mg","N_PROT": "Protein_g",
}
df.rename(columns=COLUMN_MAP, inplace=True)
df.to_csv(OUTPUT_DIR / "knhanes_fpg_preprocessed.csv", index=False, encoding="utf-8")
log.info("Preprocessed dataset saved: %d rows × %d cols", *df.shape)


2026-04-27 13:43:13,130 | INFO | Step 1 — Drop missing FPG: 20191 → 18039 (removed 2152)
2026-04-27 13:43:13,133 | WARNING | Step 2 — 'HE_DMdr' not found in this dataset (column absent in selected survey years). Skipping exclusion.
2026-04-27 13:43:13,165 | INFO | Step 4 — FPG outlier removal: 16678 → 16677 (removed 1)
2026-04-27 13:43:13,657 | INFO | Preprocessed dataset saved: 16677 rows × 38 cols


In [5]:
# 실제 컬럼 확인
print("HE_DMdr 존재 여부:", "HE_DMdr" in df_raw.columns)
print("HE_glu 존재 여부:",  "HE_glu"  in df_raw.columns)

# HE_DMdr과 유사한 컬럼명 찾기
similar = [c for c in df_raw.columns if "DM" in c or "dm" in c or "당뇨" in c]
print("DM 관련 컬럼:", similar)

# 전체 컬럼 확인
print("\n전체 컬럼 목록:")
print(sorted(df_raw.columns.tolist()))

HE_DMdr 존재 여부: False
HE_glu 존재 여부: True
DM 관련 컬럼: []

전체 컬럼 목록:
['BD1_11', 'BD2_1', 'BE3_71', 'BE3_75', 'BE3_81', 'BE3_91', 'BH1', 'BO1_1', 'BO1_2', 'BO1_3', 'BP1', 'BS3_1', 'HE_BMI', 'HE_glu', 'HE_obe', 'HE_wc', 'HE_wt', 'ID', 'L_BR_FQ', 'N_CHO', 'N_EN', 'N_FAT', 'N_K', 'N_NA', 'N_PROT', 'N_SFA', 'N_SUGAR', 'N_TDF', 'age', 'edu', 'ho_incm', 'incm', 'mh_stress', 'pa_aerobic', 'sex', 'year']


## 4. Feature Definition & Final Dataset

In [6]:
CAT_FEATURES = [
    "ObesityStatus",    "WeightChangeStatus", "WeightLossAmount", "WeightGainAmount",
    "DrinkingFrequency","DrinkingAmount",      "SmokingStatus",
    "VigorousAct_Work", "VigorousAct_Leisure","ModerateAct_Work",
    "WalkingActivity",  "AerobicRate",         "BreakfastFreq",
    "StressLevel",      "StressAwareness",
    "IncomeQuartile",   "HouseholdIncome",     "EducationLevel", "HealthScreening",
]
NUM_FEATURES = [
    "BMI", "WaistCirc", "Weight",
    "Energy_kcal", "Carb_g",  "Sugar_g",  "Sodium_mg",
    "Fat_g",       "SatFat_g","Fiber_g",   "Potassium_mg", "Protein_g",
]
X_FEATURES = NUM_FEATURES + CAT_FEATURES

required = X_FEATURES + ["FPG", "GlucoseStage", "Sex", "AgeGroup", "SurveyYear"]
df_final = df[[c for c in required if c in df.columns]].copy()
for col in df_final.columns:
    if col != "GlucoseStage":
        df_final[col] = pd.to_numeric(df_final[col], errors="coerce").fillna(0)

log.info("df_final shape: %s", df_final.shape)
print(df_final[["FPG","GlucoseStage","Sex","AgeGroup"]].head())


2026-04-27 13:43:13,771 | INFO | df_final shape: (16677, 36)


    FPG GlucoseStage  Sex  AgeGroup
0  94.0       normal  2.0       1.0
1  84.0       normal  1.0       0.0
2  87.0       normal  2.0       0.0
3  87.0       normal  1.0       2.0
4  91.0       normal  2.0       1.0


## 5. Table 1 — Descriptive Statistics by Group

In [7]:
rows = []
for grp, cfg in GROUP_CONFIG.items():
    sub = df_final[
        (df_final["AgeGroup"] == cfg["age_group"]) &
        (df_final["Sex"]      == cfg["sex_code"])
    ]
    n = len(sub)
    vc = sub["GlucoseStage"].value_counts(normalize=True) * 100
    rows.append({
        "Group":         GROUP_LABELS[grp],
        "n":             n,
        "FPG_Mean":      round(sub["FPG"].mean(), 2),
        "FPG_SD":        round(sub["FPG"].std(),  2),
        "FPG_Median":    round(sub["FPG"].median(), 1),
        "Normal_pct":    round(vc.get("normal",   0), 1),
        "IFG_pct":       round(vc.get("ifg",      0), 1),
        "Diabetes_pct":  round(vc.get("diabetes", 0), 1),
    })

table1 = pd.DataFrame(rows)
table1.to_csv(OUTPUT_DIR / "table1_descriptive_stats.csv", index=False, encoding="utf-8")
print(table1.to_string(index=False))


             Group    n  FPG_Mean  FPG_SD  FPG_Median  Normal_pct  IFG_pct  Diabetes_pct
        Young Male 1745     94.30   15.94        92.0        80.5     17.7           1.8
      Young Female 2075     90.17   13.50        89.0        91.3      7.6           1.1
  Middle-aged Male 3226    106.67   25.67       100.0        48.5     39.1          12.4
Middle-aged Female 4432     98.91   20.65        95.0        68.0     26.2           5.8
      Elderly Male 2275    109.72   26.02       103.0        40.9     41.8          17.2
    Elderly Female 2924    105.00   21.90        99.0        51.1     37.3          11.6


## 6. Figure 1 — FPG Distribution by Demographic Group

In [8]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharey=False)
axes = axes.flatten()

stage_colors = {"normal": "#4CAF50", "ifg": "#FF9800", "diabetes": "#F44336"}
stage_labels = {"normal": "Normal", "ifg": "IFG", "diabetes": "Diabetes"}

vline_kwargs = dict(linestyle="--", linewidth=1.2, alpha=0.7)

for ax, (grp, cfg) in zip(axes, GROUP_CONFIG.items()):
    sub = df_final[
        (df_final["AgeGroup"] == cfg["age_group"]) &
        (df_final["Sex"]      == cfg["sex_code"])
    ]["FPG"]

    # Histogram
    ax.hist(sub, bins=50, color=GROUP_COLORS[grp], alpha=0.75,
            edgecolor="white", linewidth=0.4)

    # Stage boundary lines
    ax.axvline(100, color="#FF9800", **vline_kwargs, label="IFG boundary (100)")
    ax.axvline(126, color="#F44336", **vline_kwargs, label="Diabetes boundary (126)")

    # Annotation
    ax.set_title(GROUP_LABELS[grp], fontsize=12, fontweight="bold")
    ax.set_xlabel("Fasting Plasma Glucose (mg/dL)", fontsize=9)
    ax.set_ylabel("Count", fontsize=9)
    ax.annotate(
        f"n = {len(sub):,}\nMean = {sub.mean():.1f}\nSD = {sub.std():.1f}",
        xy=(0.97, 0.97), xycoords="axes fraction",
        ha="right", va="top", fontsize=8,
        bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8),
    )

# Shared legend
handles = [
    mpatches.Patch(color="#FF9800", alpha=0.7, label="IFG boundary (100 mg/dL)"),
    mpatches.Patch(color="#F44336", alpha=0.7, label="Diabetes boundary (126 mg/dL)"),
]
fig.legend(handles=handles, loc="lower center", ncol=2, fontsize=9,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle("Fasting Plasma Glucose Distribution by Demographic Group\n(KNHANES; n = 27,934)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()

fig.savefig(OUTPUT_DIR / "fig1_fpg_distribution_by_group.png",
            dpi=DPI, bbox_inches="tight")
plt.show()
log.info("Figure 1 saved.")


2026-04-27 13:43:15,775 | INFO | Figure 1 saved.


## 7. Model Training Functions

In [9]:
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Return RMSE, MAE, R² for a regression prediction."""
    return {
        "RMSE": round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 4),
        "MAE":  round(float(mean_absolute_error(y_true, y_pred)), 4),
        "R2":   round(float(r2_score(y_true, y_pred)), 4),
    }


def train_model(X_train, X_val, y_train, y_val,
                algorithm: str, n_trials: int = 25) -> tuple:
    """
    Train a single algorithm with Optuna hyperparameter optimisation.

    Parameters
    ----------
    X_train, X_val : pd.DataFrame
    y_train, y_val : pd.Series
    algorithm      : one of ALGORITHMS
    n_trials       : Optuna optimisation trials

    Returns
    -------
    (fitted_model, metrics_dict, best_params_dict)
    """
    def _objective_rf(trial):
        m = RandomForestRegressor(
            n_estimators      = trial.suggest_int("n_estimators", 100, 500),
            max_depth         = trial.suggest_int("max_depth", 3, 15),
            min_samples_split = trial.suggest_int("min_samples_split", 2, 10),
            random_state=SEED, n_jobs=-1,
        )
        m.fit(X_train, y_train)
        return -mean_squared_error(y_val, m.predict(X_val))

    def _objective_lgbm(trial):
        m = lgb.LGBMRegressor(
            n_estimators     = trial.suggest_int("n_estimators", 100, 500),
            max_depth        = trial.suggest_int("max_depth", 3, 7),
            learning_rate    = trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            subsample        = trial.suggest_float("subsample", 0.7, 1.0),
            colsample_bytree = trial.suggest_float("colsample_bytree", 0.7, 1.0),
            random_state=SEED, n_jobs=-1, verbose=-1,
        )
        m.fit(X_train, y_train)
        return -mean_squared_error(y_val, m.predict(X_val))

    def _objective_xgb(trial):
        m = xgb.XGBRegressor(
            n_estimators     = trial.suggest_int("n_estimators", 100, 500),
            max_depth        = trial.suggest_int("max_depth", 3, 7),
            learning_rate    = trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            subsample        = trial.suggest_float("subsample", 0.7, 1.0),
            colsample_bytree = trial.suggest_float("colsample_bytree", 0.7, 1.0),
            random_state=SEED, tree_method="hist", verbosity=0,
        )
        m.fit(X_train, y_train)
        return -mean_squared_error(y_val, m.predict(X_val))

    def _objective_mlp(trial):
        m = MLPRegressor(
            hidden_layer_sizes = tuple(
                [trial.suggest_int("n_units", 32, 256)] *
                trial.suggest_int("n_layers", 1, 3)
            ),
            alpha              = trial.suggest_float("alpha", 1e-5, 1e-2, log=True),
            learning_rate_init = trial.suggest_float("lr_init", 1e-4, 1e-2, log=True),
            max_iter=300, random_state=SEED,
        )
        m.fit(X_train, y_train)
        return -mean_squared_error(y_val, m.predict(X_val))

    best_params = {}

    if algorithm == "LR":
        model = LinearRegression()
        model.fit(X_train, y_train)

    elif algorithm == "Ridge":
        study = optuna.create_study(direction="maximize",
                                    sampler=optuna.samplers.TPESampler(seed=SEED))
        study.optimize(
            lambda t: -mean_squared_error(
                y_val,
                Ridge(alpha=t.suggest_float("alpha", 1e-3, 100.0, log=True),
                      random_state=SEED).fit(X_train, y_train).predict(X_val)
            ), n_trials=n_trials, show_progress_bar=False,
        )
        best_params = study.best_params
        model = Ridge(**best_params, random_state=SEED)
        model.fit(X_train, y_train)

    elif algorithm == "RF":
        study = optuna.create_study(direction="maximize",
                                    sampler=optuna.samplers.TPESampler(seed=SEED))
        study.optimize(_objective_rf, n_trials=n_trials, show_progress_bar=False)
        best_params = study.best_params
        model = RandomForestRegressor(**best_params, random_state=SEED, n_jobs=-1)
        model.fit(X_train, y_train)

    elif algorithm == "LGBM":
        study = optuna.create_study(direction="maximize",
                                    sampler=optuna.samplers.TPESampler(seed=SEED))
        study.optimize(_objective_lgbm, n_trials=n_trials, show_progress_bar=False)
        best_params = study.best_params
        model = lgb.LGBMRegressor(**best_params, random_state=SEED,
                                   n_jobs=-1, verbose=-1)
        model.fit(X_train, y_train)

    elif algorithm == "XGB":
        study = optuna.create_study(direction="maximize",
                                    sampler=optuna.samplers.TPESampler(seed=SEED))
        study.optimize(_objective_xgb, n_trials=n_trials, show_progress_bar=False)
        best_params = study.best_params
        model = xgb.XGBRegressor(**best_params, random_state=SEED,
                                   tree_method="hist", verbosity=0)
        model.fit(X_train, y_train)

    elif algorithm == "MLP":
        study = optuna.create_study(direction="maximize",
                                    sampler=optuna.samplers.TPESampler(seed=SEED))
        study.optimize(_objective_mlp, n_trials=n_trials, show_progress_bar=False)
        bp = study.best_params
        best_params = bp
        model = MLPRegressor(
            hidden_layer_sizes = tuple([bp["n_units"]] * bp["n_layers"]),
            alpha              = bp["alpha"],
            learning_rate_init = bp["lr_init"],
            max_iter=300, random_state=SEED,
        )
        model.fit(X_train, y_train)

    else:
        raise ValueError(f"Unsupported algorithm: {algorithm}")

    metrics = compute_metrics(y_val, model.predict(X_val))
    return model, metrics, best_params

log.info("Model training functions defined.")


2026-04-27 13:43:15,828 | INFO | Model training functions defined.


## 8. Stratified Training — 6 Groups × 6 Algorithms

In [10]:
all_results    = {}   # {group: {algo: metrics}}
best_models    = {}   # {group: fitted_model}
best_algo_name = {}   # {group: str}
best_params_store = {}

for grp, cfg in GROUP_CONFIG.items():
    log.info("=" * 55)
    log.info("Group: %s  (n=%d)", GROUP_LABELS[grp], sum(
        (df_final["AgeGroup"] == cfg["age_group"]) &
        (df_final["Sex"]      == cfg["sex_code"])
    ))

    df_g = df_final[
        (df_final["AgeGroup"] == cfg["age_group"]) &
        (df_final["Sex"]      == cfg["sex_code"])
    ].copy()

    X = df_g[X_FEATURES]
    y = df_g["FPG"]
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.20, random_state=SEED
    )

    all_results[grp]    = {}
    best_rmse_grp       = float("inf")
    best_model_grp      = None
    best_algo_grp       = None

    for algo in ALGORITHMS:
        try:
            model, metrics, bp = train_model(X_train, X_val, y_train, y_val, algo)
            all_results[grp][algo] = metrics
            best_params_store[(grp, algo)] = bp
            is_baseline = algo in BASELINE_ALGOS
            tag = " [baseline]" if is_baseline else ""
            log.info("  %-6s | RMSE=%.4f  MAE=%.4f  R²=%.4f%s",
                     algo, metrics["RMSE"], metrics["MAE"], metrics["R2"], tag)

            # Best model: exclude baseline algorithms
            if (not is_baseline) and (metrics["RMSE"] < best_rmse_grp):
                best_rmse_grp  = metrics["RMSE"]
                best_model_grp = model
                best_algo_grp  = algo

        except Exception as exc:
            log.warning("  %-6s | FAILED: %s", algo, exc)
            all_results[grp][algo] = None

    best_models[grp]    = best_model_grp
    best_algo_name[grp] = best_algo_grp
    log.info("  >>> Best model: %s (RMSE=%.4f)", best_algo_grp, best_rmse_grp)

log.info("Training complete for all groups.")


2026-04-27 13:43:15,873 | INFO | =======================================================
2026-04-27 13:43:15,876 | INFO | Group: Young Male  (n=1745)
2026-04-27 13:43:15,959 | INFO |   LR     | RMSE=18.4774  MAE=8.4897  R²=0.0636 [baseline]
2026-04-27 13:43:16,159 | INFO |   Ridge  | RMSE=18.4774  MAE=8.4897  R²=0.0636 [baseline]
2026-04-27 13:43:37,334 | INFO |   RF     | RMSE=18.4761  MAE=8.7800  R²=0.0637
2026-04-27 13:43:42,027 | INFO |   LGBM   | RMSE=18.3860  MAE=8.5718  R²=0.0728
2026-04-27 13:43:59,657 | INFO |   XGB    | RMSE=18.6202  MAE=8.9977  R²=0.0490
2026-04-27 13:44:53,865 | INFO |   MLP    | RMSE=19.6943  MAE=11.3384  R²=-0.0638
2026-04-27 13:44:53,867 | INFO |   >>> Best model: LGBM (RMSE=18.3860)
2026-04-27 13:44:53,867 | INFO | =======================================================
2026-04-27 13:44:53,871 | INFO | Group: Young Female  (n=2075)
2026-04-27 13:44:54,032 | INFO |   LR     | RMSE=15.8536  MAE=7.2326  R²=0.0823 [baseline]
2026-04-27 13:44:54,366 | INFO |

## 9. 5-Fold Cross-Validation

In [11]:
def run_cross_validation(df_group: pd.DataFrame, algorithm: str,
                         group_name: str, n_splits: int = 5) -> dict:
    """Run k-fold CV for a given group and algorithm, reusing best hyperparams."""
    X  = df_group[X_FEATURES]
    y  = df_group["FPG"]
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    bp = best_params_store.get((group_name, algorithm), {})

    fold_rmse = []
    for tr_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        if algorithm == "LR":
            m = LinearRegression()
        elif algorithm == "Ridge":
            m = Ridge(alpha=bp.get("alpha", 1.0), random_state=SEED)
        elif algorithm == "RF":
            m = RandomForestRegressor(
                **{k: v for k, v in bp.items()
                   if k in ["n_estimators","max_depth","min_samples_split"]},
                random_state=SEED, n_jobs=-1)
        elif algorithm == "LGBM":
            m = lgb.LGBMRegressor(
                **{k: v for k, v in bp.items()
                   if k in ["n_estimators","max_depth","learning_rate",
                             "subsample","colsample_bytree"]},
                random_state=SEED, n_jobs=-1, verbose=-1)
        elif algorithm == "XGB":
            m = xgb.XGBRegressor(
                **{k: v for k, v in bp.items()
                   if k in ["n_estimators","max_depth","learning_rate",
                             "subsample","colsample_bytree"]},
                random_state=SEED, tree_method="hist", verbosity=0)
        elif algorithm == "MLP":
            m = MLPRegressor(
                hidden_layer_sizes=tuple([bp.get("n_units",128)] * bp.get("n_layers",2)),
                alpha=bp.get("alpha",1e-4),
                learning_rate_init=bp.get("lr_init",1e-3),
                max_iter=300, random_state=SEED)
        else:
            continue

        m.fit(X_tr, y_tr)
        fold_rmse.append(float(np.sqrt(mean_squared_error(y_val, m.predict(X_val)))))

    return {
        "CV_RMSE_mean": round(np.mean(fold_rmse), 4),
        "CV_RMSE_std":  round(np.std(fold_rmse),  4),
    }


cv_results = {}
for grp, cfg in GROUP_CONFIG.items():
    df_g  = df_final[
        (df_final["AgeGroup"] == cfg["age_group"]) &
        (df_final["Sex"]      == cfg["sex_code"])
    ].copy()
    algo  = best_algo_name[grp]
    cv    = run_cross_validation(df_g, algo, grp)
    lr_cv = run_cross_validation(df_g, "LR", grp)
    cv_results[grp] = {**cv, "LR_CV_RMSE_mean": lr_cv["CV_RMSE_mean"]}
    ho = all_results[grp].get(algo, {})
    gap = abs(cv["CV_RMSE_mean"] - ho.get("RMSE", cv["CV_RMSE_mean"]))
    verdict = "OK" if gap < 1.0 else ("Caution" if gap < 2.0 else "Risk")
    log.info("[%s] %s | HO=%.4f  CV=%.4f±%.4f  Gap=%.4f  [G2-a: %s]",
             grp, algo, ho.get("RMSE",0), cv["CV_RMSE_mean"],
             cv["CV_RMSE_std"], gap, verdict)

cv_df = pd.DataFrame([
    {
        "Group":          GROUP_LABELS[grp],
        "Best_Algorithm": best_algo_name[grp],
        "HO_RMSE":        all_results[grp].get(best_algo_name[grp],{}).get("RMSE",""),
        "HO_MAE":         all_results[grp].get(best_algo_name[grp],{}).get("MAE",""),
        "HO_R2":          all_results[grp].get(best_algo_name[grp],{}).get("R2",""),
        "CV_RMSE_mean":   cv_results[grp]["CV_RMSE_mean"],
        "CV_RMSE_std":    cv_results[grp]["CV_RMSE_std"],
    }
    for grp in GROUP_CONFIG
])
cv_df.to_csv(OUTPUT_DIR / "table2_cv_results.csv", index=False, encoding="utf-8")
log.info("CV results saved.")


2026-04-27 13:54:15,224 | INFO | [Young_Male] LGBM | HO=18.3860  CV=15.1107±3.9770  Gap=3.2753  [G2-a: Risk]
2026-04-27 13:54:15,950 | INFO | [Young_Female] LGBM | HO=15.8624  CV=12.8839±2.4338  Gap=2.9785  [G2-a: Risk]
2026-04-27 13:54:17,262 | INFO | [Middle_Male] LGBM | HO=25.9668  CV=25.1956±2.0116  Gap=0.7712  [G2-a: OK]
2026-04-27 13:54:19,593 | INFO | [Middle_Female] XGB | HO=17.3049  CV=19.7553±1.7777  Gap=2.4504  [G2-a: Risk]
2026-04-27 13:54:20,158 | INFO | [Elderly_Male] LGBM | HO=21.4421  CV=25.5796±3.2447  Gap=4.1375  [G2-a: Risk]
2026-04-27 13:54:28,363 | INFO | [Elderly_Female] RF | HO=20.1553  CV=21.8464±1.0217  Gap=1.6911  [G2-a: Caution]
2026-04-27 13:54:28,376 | INFO | CV results saved.


## 10. Temporal Validation (G2-b Robustness)

In [12]:
# Train on earlier years → test on most recent year
# Assesses distributional stability over time (Varma & Simon, 2006)

YEAR_TRAIN = [2022, 2023]   # adjust to your actual survey years
YEAR_TEST  = 2024

temporal_results = {}
for grp, cfg in GROUP_CONFIG.items():
    df_g = df_final[
        (df_final["AgeGroup"] == cfg["age_group"]) &
        (df_final["Sex"]      == cfg["sex_code"])
    ].copy()

    train_mask = df_g["SurveyYear"].isin(YEAR_TRAIN)
    test_mask  = df_g["SurveyYear"] == YEAR_TEST

    if train_mask.sum() < 30 or test_mask.sum() < 10:
        log.warning("[%s] Insufficient temporal data — skipping", grp)
        temporal_results[grp] = None
        continue

    X_tr  = df_g.loc[train_mask, X_FEATURES]
    y_tr  = df_g.loc[train_mask, "FPG"]
    X_te  = df_g.loc[test_mask,  X_FEATURES]
    y_te  = df_g.loc[test_mask,  "FPG"]

    # Retrain best algorithm on training years
    algo = best_algo_name[grp]
    bp   = best_params_store.get((grp, algo), {})

    if algo == "RF":
        m = RandomForestRegressor(
            **{k: v for k, v in bp.items()
               if k in ["n_estimators","max_depth","min_samples_split"]},
            random_state=SEED, n_jobs=-1)
    elif algo == "LGBM":
        m = lgb.LGBMRegressor(
            **{k: v for k, v in bp.items()
               if k in ["n_estimators","max_depth","learning_rate",
                         "subsample","colsample_bytree"]},
            random_state=SEED, n_jobs=-1, verbose=-1)
    elif algo == "XGB":
        m = xgb.XGBRegressor(
            **{k: v for k, v in bp.items()
               if k in ["n_estimators","max_depth","learning_rate",
                         "subsample","colsample_bytree"]},
            random_state=SEED, tree_method="hist", verbosity=0)
    else:
        m = LinearRegression()

    m.fit(X_tr, y_tr)
    ho_rmse   = all_results[grp].get(algo, {}).get("RMSE", float("nan"))
    temp_rmse = float(np.sqrt(mean_squared_error(y_te, m.predict(X_te))))
    delta     = round(temp_rmse - ho_rmse, 4)
    direction = "Degraded" if delta > 0 else "Improved"
    verdict   = "Risk" if delta > 0 else "OK"

    temporal_results[grp] = {
        "HO_RMSE": ho_rmse, "Temporal_RMSE": round(temp_rmse, 4),
        "Delta": delta, "Direction": direction, "G2b_Verdict": verdict,
    }
    log.info("[%s] HO=%.4f  Temporal=%.4f  Δ=%+.4f  [%s] → G2-b: %s",
             grp, ho_rmse, temp_rmse, delta, direction, verdict)

temp_df = pd.DataFrame([
    {"Group": GROUP_LABELS[g], **(v or {})}
    for g, v in temporal_results.items()
])
temp_df.to_csv(OUTPUT_DIR / "table3_temporal_validation.csv",
               index=False, encoding="utf-8")
log.info("Temporal validation saved.")


2026-04-27 13:54:28,554 | INFO | [Young_Male] HO=18.3860  Temporal=19.6918  Δ=+1.3058  [Degraded] → G2-b: Risk
2026-04-27 13:54:28,666 | INFO | [Young_Female] HO=15.8624  Temporal=10.2477  Δ=-5.6147  [Improved] → G2-b: OK
2026-04-27 13:54:28,880 | INFO | [Middle_Male] HO=25.9668  Temporal=23.9803  Δ=-1.9865  [Improved] → G2-b: OK
2026-04-27 13:54:29,247 | INFO | [Middle_Female] HO=17.3049  Temporal=19.0526  Δ=+1.7477  [Degraded] → G2-b: Risk
2026-04-27 13:54:29,323 | INFO | [Elderly_Male] HO=21.4421  Temporal=24.3774  Δ=+2.9353  [Degraded] → G2-b: Risk
2026-04-27 13:54:30,757 | INFO | [Elderly_Female] HO=20.1553  Temporal=21.3671  Δ=+1.2118  [Degraded] → G2-b: Risk
2026-04-27 13:54:30,769 | INFO | Temporal validation saved.


## 11. Figure 2 — Algorithm Performance Heatmap

In [13]:
# Build RMSE matrix: rows = groups, cols = algorithms
rmse_matrix = pd.DataFrame(index=list(GROUP_CONFIG.keys()), columns=ALGORITHMS)
for grp in GROUP_CONFIG:
    for algo in ALGORITHMS:
        m = all_results.get(grp, {}).get(algo)
        rmse_matrix.loc[grp, algo] = m["RMSE"] if m else np.nan
rmse_matrix = rmse_matrix.astype(float)
rmse_matrix.index = [GROUP_LABELS[g] for g in rmse_matrix.index]

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    rmse_matrix, annot=True, fmt=".2f", cmap="YlOrRd",
    linewidths=0.5, linecolor="white",
    cbar_kws={"label": "RMSE (mg/dL)"},
    ax=ax,
)
ax.set_title("Hold-Out RMSE by Demographic Group and Algorithm\n"
             "(lower = better; ★ = best per group)",
             fontsize=12, fontweight="bold", pad=12)
ax.set_xlabel("Algorithm", fontsize=11)
ax.set_ylabel("Demographic Group", fontsize=11)

# Star the best model per group
for i, grp in enumerate(GROUP_CONFIG):
    best_a = best_algo_name.get(grp)
    if best_a and best_a in ALGORITHMS:
        j = ALGORITHMS.index(best_a)
        ax.text(j + 0.5, i + 0.15, "★", ha="center", va="center",
                color="navy", fontsize=14, fontweight="bold")

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "fig2_rmse_heatmap.png", dpi=DPI, bbox_inches="tight")
plt.show()
log.info("Figure 2 saved.")


2026-04-27 13:54:31,362 | INFO | Figure 2 saved.


## 12. SHAP Analysis + HHI Concentration (G3)

In [14]:
def compute_hhi_bootstrap(shap_vals: np.ndarray,
                          n_bootstrap: int = 1000,
                          ci: float = 0.95) -> dict:
    """
    Compute SHAP HHI with bootstrap confidence interval.
    HHI = Σ sᵢ² where sᵢ = normalised mean |SHAP| share of feature i.
    Thresholds (G3): < 0.18 OK | 0.18-0.25 Caution | > 0.25 Risk
    """
    def _hhi(sv):
        mean_abs = np.mean(np.abs(sv), axis=0)
        total    = mean_abs.sum()
        if total == 0: return 0.0
        shares   = mean_abs / total
        return float(np.sum(shares ** 2))

    observed = _hhi(shap_vals)
    boots    = [_hhi(shap_vals[np.random.randint(0, len(shap_vals), len(shap_vals))])
                for _ in range(n_bootstrap)]
    alpha    = (1 - ci) / 2
    return {
        "HHI":    round(observed, 4),
        "CI_lo":  round(np.percentile(boots, alpha * 100), 4),
        "CI_hi":  round(np.percentile(boots, (1 - alpha) * 100), 4),
        "G3_verdict": ("Risk" if observed > 0.25
                       else "Caution" if observed > 0.18
                       else "OK"),
    }


shap_results = {}
hhi_results  = {}

for grp, cfg in GROUP_CONFIG.items():
    df_g = df_final[
        (df_final["AgeGroup"] == cfg["age_group"]) &
        (df_final["Sex"]      == cfg["sex_code"])
    ].copy()
    model = best_models[grp]
    algo  = best_algo_name[grp]
    if model is None:
        continue

    X     = df_g[X_FEATURES]
    y     = df_g["FPG"]
    _, X_val, _, _ = train_test_split(X, y, test_size=0.20, random_state=SEED)
    X_s   = X_val.sample(min(500, len(X_val)), random_state=SEED)

    log.info("SHAP: [%s] (%s)", grp, algo)

    # ── SHAP values ──────────────────────────────────────────────────────────
    if algo in ("RF", "LGBM", "XGB"):
        explainer   = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_s)
    else:
        bg        = shap.sample(X_s, min(100, len(X_s)))
        explainer = shap.KernelExplainer(model.predict, bg)
        shap_values = explainer.shap_values(X_s.iloc[:50])
        X_s         = X_s.iloc[:50]

    # ── HHI ──────────────────────────────────────────────────────────────────
    hhi = compute_hhi_bootstrap(shap_values)
    hhi_results[grp] = hhi
    log.info("  HHI=%.4f [%.4f, %.4f]  G3=%s",
             hhi["HHI"], hhi["CI_lo"], hhi["CI_hi"], hhi["G3_verdict"])

    # ── SHAP Summary Plot (English) ───────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 6))
    shap.summary_plot(shap_values, X_s, plot_type="dot",
                      show=False, max_display=15, color_bar=True)
    ax = plt.gca()
    ax.set_title(f"SHAP Feature Importance — {GROUP_LABELS[grp]}\n"
                 f"({algo}; HHI = {hhi['HHI']:.3f}  G3: {hhi['G3_verdict']})",
                 fontsize=11, fontweight="bold")
    ax.set_xlabel("SHAP Value (impact on FPG prediction, mg/dL)", fontsize=10)
    plt.tight_layout()
    fname = OUTPUT_DIR / f"fig_shap_summary_{grp}.png"
    plt.savefig(fname, dpi=DPI, bbox_inches="tight")
    plt.close()
    log.info("  Saved: %s", fname)

    shap_results[grp] = {"shap_values": shap_values, "X_sample": X_s,
                          "explainer": explainer}

# ── HHI Summary Table ─────────────────────────────────────────────────────────
hhi_df = pd.DataFrame([
    {
        "Group":       GROUP_LABELS[g],
        "Algorithm":   best_algo_name[g],
        "SHAP_HHI":    v["HHI"],
        "CI_95_lower": v["CI_lo"],
        "CI_95_upper": v["CI_hi"],
        "G3_verdict":  v["G3_verdict"],
    }
    for g, v in hhi_results.items()
])
hhi_df.to_csv(OUTPUT_DIR / "table4_shap_hhi.csv", index=False, encoding="utf-8")
print(hhi_df.to_string(index=False))


2026-04-27 13:54:31,414 | INFO | SHAP: [Young_Male] (LGBM)
2026-04-27 13:54:31,578 | INFO |   HHI=0.1389 [0.1323, 0.1463]  G3=OK
2026-04-27 13:54:32,463 | INFO |   Saved: outputs\fig_shap_summary_Young_Male.png
2026-04-27 13:54:32,476 | INFO | SHAP: [Young_Female] (LGBM)
2026-04-27 13:54:32,674 | INFO |   HHI=0.1515 [0.1382, 0.1664]  G3=OK
2026-04-27 13:54:33,580 | INFO |   Saved: outputs\fig_shap_summary_Young_Female.png
2026-04-27 13:54:33,590 | INFO | SHAP: [Middle_Male] (LGBM)
2026-04-27 13:54:33,969 | INFO |   HHI=0.0742 [0.0716, 0.0772]  G3=OK
2026-04-27 13:54:34,920 | INFO |   Saved: outputs\fig_shap_summary_Middle_Male.png
2026-04-27 13:54:34,927 | INFO | SHAP: [Middle_Female] (XGB)
2026-04-27 13:54:35,224 | INFO |   HHI=0.0930 [0.0886, 0.0978]  G3=OK
2026-04-27 13:54:36,244 | INFO |   Saved: outputs\fig_shap_summary_Middle_Female.png
2026-04-27 13:54:36,253 | INFO | SHAP: [Elderly_Male] (LGBM)
2026-04-27 13:54:36,458 | INFO |   HHI=0.0940 [0.0879, 0.1015]  G3=OK
2026-04-27 13:

             Group Algorithm  SHAP_HHI  CI_95_lower  CI_95_upper G3_verdict
        Young Male      LGBM    0.1389       0.1323       0.1463         OK
      Young Female      LGBM    0.1515       0.1382       0.1664         OK
  Middle-aged Male      LGBM    0.0742       0.0716       0.0772         OK
Middle-aged Female       XGB    0.0930       0.0886       0.0978         OK
      Elderly Male      LGBM    0.0940       0.0879       0.1015         OK
    Elderly Female        RF    0.1045       0.1000       0.1098         OK


## 13. Save Artefacts (→ Notebooks 02–05)

In [15]:
# Save all artefacts needed by downstream notebooks
df_final.to_parquet(OUTPUT_DIR / "df_final.parquet", index=False)
joblib.dump(all_results,       OUTPUT_DIR / "all_results.pkl")
joblib.dump(best_models,       OUTPUT_DIR / "best_models.pkl")
joblib.dump(best_algo_name,    OUTPUT_DIR / "best_algo_name.pkl")
joblib.dump(best_params_store, OUTPUT_DIR / "best_params_store.pkl")
joblib.dump(shap_results,      OUTPUT_DIR / "shap_results.pkl")
joblib.dump(cv_df,             OUTPUT_DIR / "cv_df.pkl")
joblib.dump(temporal_results,  OUTPUT_DIR / "temporal_results.pkl")
joblib.dump(hhi_results,       OUTPUT_DIR / "hhi_results.pkl")

for grp in GROUP_CONFIG:
    mdl = best_models.get(grp)
    if mdl is not None:
        joblib.dump(mdl, OUTPUT_DIR / f"model_{grp}.pkl")

log.info("=" * 55)
log.info("Notebook 01 complete. Outputs in: %s/", OUTPUT_DIR)
log.info("=" * 55)
for f in sorted(OUTPUT_DIR.iterdir()):
    log.info("  %s", f.name)


2026-04-27 13:54:45,723 | INFO | =======================================================
2026-04-27 13:54:45,725 | INFO | Notebook 01 complete. Outputs in: outputs/
2026-04-27 13:54:45,726 | INFO | =======================================================
2026-04-27 13:54:45,728 | INFO |   all_results.pkl
2026-04-27 13:54:45,730 | INFO |   best_algo_name.pkl
2026-04-27 13:54:45,733 | INFO |   best_models.pkl
2026-04-27 13:54:45,735 | INFO |   best_params_store.pkl
2026-04-27 13:54:45,736 | INFO |   cv_df.pkl
2026-04-27 13:54:45,739 | INFO |   df_final.parquet
2026-04-27 13:54:45,740 | INFO |   fig1_fpg_distribution_by_group.png
2026-04-27 13:54:45,741 | INFO |   fig2_rmse_heatmap.png
2026-04-27 13:54:45,742 | INFO |   fig_shap_summary_Elderly_Female.png
2026-04-27 13:54:45,742 | INFO |   fig_shap_summary_Elderly_Male.png
2026-04-27 13:54:45,743 | INFO |   fig_shap_summary_Middle_Female.png
2026-04-27 13:54:45,744 | INFO |   fig_shap_summary_Middle_Male.png
2026-04-27 13:54:45,745 | INFO 